In [1]:
import os
import warnings
from pathlib import Path

import librosa
import torch
from IPython.display import Audio
from dotenv import load_dotenv
from pyannote.audio import Pipeline

from src.ASRData import ASRData

load_dotenv()

/home/dom/GitRepos/multilingual-transcription/.venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
SAMPLE_RATE = 16000

In [3]:
pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1", token=os.environ["HUGGINGFACE_TOKEN"])

In [4]:
def load_audio_as_mapping(audio_path: str|Path) -> dict:
    waveform, samplerate = librosa.load(audio_path, sr=SAMPLE_RATE)

    waveform = torch.from_numpy(waveform).unsqueeze(0).float()

    # Compatible with pyannote, and allows us to easily split waveform into smaller chunks
    audio_mapping = {
        "waveform": waveform,
        "sample_rate": samplerate,
        "channel": 0,
        "uri": audio_path.name,
    }
    return audio_mapping

In [5]:
reference_asr_data = ASRData.from_txt_file(Path("../data/example_2/asr_data.txt"))
print(reference_asr_data)

[0.03 - 2.83] Jane: Hello I'm Jane! Is this seat free?
[3.51 - 7.14] Michał: Cześć Jane, mam na imię Michał! I jasne, możesz usiąść.
[7.83 - 10.27] Jane: Thanks! It’s my first time in Kraków.
[10.97 - 13.28] Michał: Naprawdę? Jak ci się podoba miasto?
[13.89 - 16.84] Jane: I love it so far. The old town is beautiful.
[17.58 - 21.66] Michał: Stare Miasto jest bardzo popularne wśród turystów. Byłaś już na Rynku?
[22.46 - 26.04] Jane: Yes, I was there this morning. I saw St. Mary’s Basilica.
[26.74 - 30.73] Michał: O, kościół Mariacki. Codziennie grają tam hejnał z wieży.
[31.64 - 34.42] Jane: Great! And can you recommend some Polish food?
[35.28 - 38.35] Michał: Oczywiście. Powinnaś spróbować pierogów i żurku.
[39.10 - 41.51] Jane: I know pierogi! They’re delicious.
[42.54 - 43.91] Michał: A próbowałaś już pączków?
[44.73 - 45.34] Jane: Not yet. What are they?
[47.06 - 50.44] Michał: To polskie doughnuts. Bardzo słodkie, ale świetne.
[51.38 - 53.10] Jane: Sounds dangerous for my diet.
[5

In [6]:
audio_path = Path("../data/example_2/audio.mp3")
audio = load_audio_as_mapping(audio_path)

In [7]:
with warnings.catch_warnings(action="ignore"):
    diary = pipeline(audio)

In [8]:
annotation = diary.speaker_diarization

for turn, _, speaker in annotation.itertracks(yield_label=True):
    print(f"[{turn.start:.2f}s - {turn.end:.2f}s] {speaker}")

[0.03s - 0.57s] SPEAKER_00
[0.74s - 1.47s] SPEAKER_00
[1.90s - 2.83s] SPEAKER_00
[3.51s - 5.18s] SPEAKER_01
[5.58s - 7.14s] SPEAKER_01
[7.83s - 8.32s] SPEAKER_00
[8.70s - 10.27s] SPEAKER_00
[10.97s - 11.54s] SPEAKER_01
[11.94s - 13.28s] SPEAKER_01
[13.89s - 15.03s] SPEAKER_00
[15.32s - 16.84s] SPEAKER_00
[17.58s - 20.23s] SPEAKER_01
[20.57s - 21.66s] SPEAKER_01
[22.46s - 23.96s] SPEAKER_00
[24.40s - 26.04s] SPEAKER_00
[26.74s - 28.14s] SPEAKER_01
[28.74s - 30.73s] SPEAKER_01
[31.64s - 32.16s] SPEAKER_00
[32.48s - 34.42s] SPEAKER_00
[35.28s - 38.35s] SPEAKER_01
[39.10s - 40.28s] SPEAKER_00
[40.55s - 41.51s] SPEAKER_00
[42.54s - 43.91s] SPEAKER_01
[44.73s - 45.34s] SPEAKER_00
[45.46s - 46.17s] SPEAKER_00
[47.06s - 48.16s] SPEAKER_01
[48.55s - 50.44s] SPEAKER_01
[51.38s - 53.10s] SPEAKER_00
[53.79s - 56.55s] SPEAKER_01
[57.44s - 58.06s] SPEAKER_00
[58.17s - 59.11s] SPEAKER_00


In [9]:
# Slicing the audio to individual speaker segments
segments = []
previous_speaker = None

for turn, _, speaker in annotation.itertracks(yield_label=True):
    start_idx = int(turn.start * SAMPLE_RATE)
    end_idx = int(turn.end * SAMPLE_RATE)
    waveform_segment = audio["waveform"][0, start_idx:end_idx]

    if speaker != previous_speaker:
        # another speaker is now
        segments.append({
            "waveform": waveform_segment,
            "speaker": speaker,
            "start": turn.start,
            "end": turn.end,
        })
    else:
        # The same speaker keeps talking
        previous_segment = segments[-1]
        merged_waveforms = torch.concat([previous_segment["waveform"], waveform_segment], dim=0)
        segments[-1]["waveform"] = merged_waveforms
        segments[-1]["end"] = turn.end

    previous_speaker = speaker

In [10]:
for segment in segments:
    print(f"[{segment["start"]:.2f}s - {segment["end"]:.2f}s] {segment["speaker"]}")

[0.03s - 2.83s] SPEAKER_00
[3.51s - 7.14s] SPEAKER_01
[7.83s - 10.27s] SPEAKER_00
[10.97s - 13.28s] SPEAKER_01
[13.89s - 16.84s] SPEAKER_00
[17.58s - 21.66s] SPEAKER_01
[22.46s - 26.04s] SPEAKER_00
[26.74s - 30.73s] SPEAKER_01
[31.64s - 34.42s] SPEAKER_00
[35.28s - 38.35s] SPEAKER_01
[39.10s - 41.51s] SPEAKER_00
[42.54s - 43.91s] SPEAKER_01
[44.73s - 46.17s] SPEAKER_00
[47.06s - 50.44s] SPEAKER_01
[51.38s - 53.10s] SPEAKER_00
[53.79s - 56.55s] SPEAKER_01
[57.44s - 59.11s] SPEAKER_00


In [11]:
Audio(segments[0]["waveform"], rate=SAMPLE_RATE)

In [12]:
Audio(segments[1]["waveform"], rate=SAMPLE_RATE)

In [13]:
Audio(segments[2]["waveform"], rate=SAMPLE_RATE)

In [14]:
Audio(segments[3]["waveform"], rate=SAMPLE_RATE)

In [15]:
Audio(segments[4]["waveform"], rate=SAMPLE_RATE)

In [16]:
Audio(segments[5]["waveform"], rate=SAMPLE_RATE)

In [17]:
Audio(segments[8]["waveform"], rate=SAMPLE_RATE)

In [18]:
Audio(segments[11]["waveform"], rate=SAMPLE_RATE)

In [19]:
Audio(segments[14]["waveform"], rate=SAMPLE_RATE)

In [20]:
Audio(segments[16]["waveform"], rate=SAMPLE_RATE)